In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  

ARQUIVO = 'Tabela_13_Meio_de_transporte.xlsx'
RACAS = ['Branca', 'Preta ou Parda', 'Indígena']
MODOS = ['A pé', 'Bicicleta', 'Motocicleta ou Mototáxi', 'Automóvel, táxi ou assemelhados', 'Transporte coletivo', 'Outros']
CORES = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']
xls = pd.ExcelFile(ARQUIVO)
aba_exemplo = next(s for s in xls.sheet_names if 'Exemplo' in s)
aba_dados = next(s for s in xls.sheet_names if 'BR GR UF MU' in s)
print('Abas:', xls.sheet_names)


In [ ]:
df = pd.read_excel(ARQUIVO, sheet_name=aba_exemplo, header=1).iloc[:len(MODOS)].copy()
df.columns = ['Meio de Transporte', *RACAS]
df['Meio de Transporte'] = MODOS  # ordem dos seis modos apresentada na planilha
for raca in RACAS:
    df[raca] = pd.to_numeric(df[raca], errors='coerce')
df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, col in enumerate(RACAS):
    axes[i].pie(df[col], labels=df['Meio de Transporte'], autopct='%1.3f%%', pctdistance=0.78,
                startangle=180, colors=CORES,
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.2})
    axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')
plt.suptitle('Meio de transporte no deslocamento ao trabalho, por cor ou raça — Brasil, 2022',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
bruto = pd.read_excel(ARQUIVO, sheet_name=aba_dados, header=None)
MODOS = ['A pé', 'Bicicleta', 'Motocicleta ou Mototáxi',
         'Automóvel, táxi ou assemelhados', 'Transporte coletivo', 'Outros']
linhas = bruto.iloc[3:].copy()
linhas = linhas[linhas.iloc[:, 0].notna()]
# Cada grupo de seis colunas contém os meios de transporte de uma raça.
partes = []
for i, raca in enumerate(RACAS):
    bloco = linhas.iloc[:, [0, *range(1 + i*6, 1 + (i+1)*6)]].copy()
    bloco.columns = ['Local', *MODOS]
    bloco = bloco.melt(id_vars='Local', var_name='Meio de Transporte', value_name='Percentual')
    bloco['Cor/Raça'] = raca
    partes.append(bloco)
long = pd.concat(partes, ignore_index=True)
long['Local'] = long['Local'].astype(str).str.strip()
long['Percentual'] = pd.to_numeric(long['Percentual'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
long.head()


In [ ]:
# Linhas 9 a 35 da aba são as 27 UFs; os municípios começam na linha 36.
estados = bruto.iloc[9:36, 0].dropna().astype(str).str.strip().tolist()
cidades = bruto.iloc[36:, 0].dropna().astype(str).str.strip().tolist()
estados_df = long[long['Local'].isin(estados)].copy()
estados_df[['Local']].drop_duplicates().sort_values('Local').reset_index(drop=True)


In [ ]:
geral = (long.groupby(['Local', 'Meio de Transporte'], as_index=False)['Percentual']
         .mean())  # NaN ('.' na planilha) é ignorado
estados_geral = geral[geral['Local'].isin(estados)]
cidades_geral = geral[geral['Local'].isin(cidades)]
coletivo_estados = estados_geral[estados_geral['Meio de Transporte'].eq('Transporte coletivo')].dropna(subset=['Percentual'])
print('Maior transporte coletivo:', coletivo_estados.loc[coletivo_estados['Percentual'].idxmax()].to_dict())
print('Menor transporte coletivo:', coletivo_estados.loc[coletivo_estados['Percentual'].idxmin()].to_dict())
print('Maior bicicleta:', cidades_geral[cidades_geral['Meio de Transporte'].eq('Bicicleta')].dropna(subset=['Percentual']).nlargest(1, 'Percentual'))
print('Maior automóvel/táxi:', cidades_geral[cidades_geral['Meio de Transporte'].eq('Automóvel, táxi ou assemelhados')].dropna(subset=['Percentual']).nlargest(1, 'Percentual'))




 Maior percentual médio de mulheres no transporte coletivo entre os estados: **Rio de Janeiro (52.23%)**.
 Menor percentual médio: **Rondônia (4.33%)**.
 Cidade com maior percentual médio usando bicicleta: **Tuiuti (SP) (100.00%)**.
 Cidade com maior percentual médio usando automóvel/táxi: **Bom Jesus do Oeste (SC) (86.05%)**.

